# Tutorial 6 — Clustering & Classification with Embeddings

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Hands-on notebook (Colab-ready; no GPU required — all of it runs on a laptop)
**Suggested duration:** 120–150 minutes
**Prerequisites:** Tutorial 0 (Ecosystem Tour), Tutorial 1 (Tokenization), Tutorial 2 (Internals). Tutorial 5 (Fine-Tuning) helps with the comparison in section 14 but is not required.

Tutorial 2 opened a transformer and looked at its hidden states. Tutorial 5 bolted a head onto one and
trained it. This notebook does something cheaper and, for a surprising number of real problems, better:
it takes the vector the model already produces and hands it to scikit-learn.

That one move buys two things at once. **Clustering** — finding the groups in a pile of text nobody has
labelled. And **classification** — assigning labels, with anywhere from zero to a few hundred examples.

By the end you will be able to:

- explain what a **sentence embedding** is, and why cosine similarity is the operation that goes with it;
- spot what embeddings are *blind* to — we will build sentences the model cannot tell apart, on purpose;
- cluster an unlabelled corpus with **k-means**, and score it honestly with **ARI** and **NMI** rather than accuracy;
- pick **k** from a silhouette curve, and say why the curve is advice and not an answer;
- draw a **t-SNE** map without drawing false conclusions from it;
- **name** clusters automatically — first with statistics, then with an LLM;
- classify with **no training data at all**, by embedding the labels themselves;
- classify with **a hundred examples** — embeddings plus logistic regression, trained in under a second;
- put every approach on one scoreboard and choose by cost, not by fashion.

The running joke of this notebook is that the fanciest model in it is used as a *feature extractor*, and
everything after that is 1990s machine learning. It is not a joke about the past. It is the point.

## 0. Mental model: text goes in, a point in space comes out

An embedding model is a transformer persuaded to end every forward pass with **one vector per text** instead
of one per token. Texts that mean similar things land near each other; texts that do not, do not. Everything
in this notebook follows from that sentence.

```text
"the cat knocked the vase over"    ──►  [ 0.03, -0.11,  0.42, ...]   384 numbers
"my kitten broke a glass"          ──►  [ 0.05, -0.09,  0.39, ...]   close by
"quarterly earnings beat guidance" ──►  [-0.31,  0.22, -0.07, ...]   elsewhere entirely
```

Once text is a vector, a toolbox opens that predates LLMs by decades and runs in milliseconds:

| You want | You use | Labelled data needed |
|---|---|---|
| Groups nobody has named yet | k-means, HDBSCAN (**clustering**) | none |
| A label from a fixed set | logistic regression on embeddings | 10–1000 examples |
| A label with nothing to train on | similarity to embedded label descriptions, or an NLI model | none |
| The best accuracy available, and plenty of data | fine-tuning (tutorial 5) | 1000+ examples |
| A label *plus* a justification in prose | prompt a generative LLM | none, but you pay per call |

Read that table as a cost curve. Fine-tuning sits at the expensive-up-front end and wins when you have the
data. Prompting a large model sits at the *recurring*-cost end and wins when you need reasoning or an
explanation. The two middle rows — this notebook — are where an embarrassing share of production text
classification actually lives, because they are nearly free and hard to beat below a thousand labels.

**One vocabulary note.** *Clustering* is unsupervised: it finds structure and gives it no names.
*Classification* is supervised: it assigns names you chose in advance. People blur the two constantly.
Section 10 is built around the difference, because "cluster 0 is the sports class" is the most common
mistake in this area.

## 1. Install the libraries

In [ ]:
!pip -q install -U sentence-transformers datasets scikit-learn matplotlib pandas

### Code walkthrough — what each one is for

- **`sentence-transformers`** — the embedding models, plus an `.encode()` layer over `transformers` that
  handles pooling, batching and normalization. Section 4 writes that layer out by hand so it stops being magic.
- **`datasets`** — the corpus loader from tutorials 0 and 5.
- **`scikit-learn`** — k-means, t-SNE, logistic regression, and every metric here. All the actual machine
  learning in this notebook lives in this package.
- **`matplotlib`**, **`pandas`** — pictures and tables.

Note what is *absent*: no `accelerate`, no `peft`, no GPU. The heaviest object we load is 90 MB.

## 2. Housekeeping: a plot style worth stealing

Four figures are coming. Rather than let matplotlib choose colours afresh each time, we fix a small palette
once. This is not decoration — the choices below are the difference between a figure a reader can decode and
one they cannot.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Four categorical hues in fixed order, checked as a set for colour-vision-deficiency
# separation: no two are confusable under protanopia or deuteranopia simulation.
SERIES  = ["#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"]   # blue, orange, aqua, violet
MARKERS = ["o", "s", "^", "D"]                           # identity is never colour alone

# One hue, light to dark, for magnitude (the similarity heatmap).
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#9ec5f4", "#3987e5", "#256abf", "#0d366b"]
)

INK, INK_SOFT, RULE = "#0b0b0b", "#52514e", "#e2e1dd"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": RULE, "axes.labelcolor": INK_SOFT, "text.color": INK,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": RULE, "grid.linewidth": 0.8,
    "font.size": 10, "figure.dpi": 110, "lines.linewidth": 2,
})

print("palette ready")

### Code walkthrough — why these, and not the defaults

**Fixed order, never cycled.** Class 0 is blue in every figure in this notebook. A colour that means one
thing in figure 3 and another in figure 9 forces the reader to re-learn the legend each time.

**Marker shape as well as colour.** Around 1 in 12 men has some form of colour-vision deficiency. A scatter
plot encoding class *only* in hue is unreadable for them — and for anyone who prints it in grey. Shape is a
second channel carrying the same information, at the cost of one argument.

**One hue for magnitude, several for identity.** The heatmap shows a *quantity*, so it gets a single hue
running light-to-dark: light is little, dark is much, and the order is obvious without a legend. Categories
get *different hues at similar lightness*, because categories have no order. The classic mistake is a rainbow
colormap on a quantity — it invents boundaries the data does not have.

**Recessive axes.** Grid and spines are pale grey so the data is the only saturated thing on the canvas. If
your gridlines are as dark as your line, you have drawn a picture of a grid.

## 3. The Sentence Zoo

Before any real dataset, we build a tiny one we understand completely: eight sentences from three obvious
topics. Small enough to check by eye.

That is the whole idea. When something surprising happens on 120,000 documents, you want to already know
what *correct* looks like on eight.

In [ ]:
from sentence_transformers import SentenceTransformer

zoo = [
    "The cat knocked the vase off the table.",        # cats
    "My kitten shredded the curtains again.",         # cats
    "Purring is a cat's way of asking for food.",     # cats
    "The rover found evidence of ancient water.",     # space
    "Three astronauts docked with the station.",      # space
    "Fold the egg whites gently into the batter.",    # cooking
    "Let the dough rest for an hour before baking.",  # cooking
    "Simmer the sauce until it thickens.",            # cooking
]

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
Z = encoder.encode(zoo, normalize_embeddings=True)

print("dimensions:", encoder.get_sentence_embedding_dimension())
print("max input :", encoder.max_seq_length, "tokens")
print("embeddings:", Z.shape, Z.dtype)
print("length of row 0:", float((Z[0] ** 2).sum()) ** 0.5)
print()
print("first 8 numbers of sentence 0:", Z[0][:8].round(3))

### Code walkthrough — the three numbers that matter

**384 dimensions.** `all-MiniLM-L6-v2` is a 6-layer, 22 M-parameter distillation of BERT, and it is the
default answer to "which embedding model?" for good reason: small, fast on CPU, and shockingly competitive.
Bigger models — `all-mpnet-base-v2` (768 dims), the `BAAI/bge-*` and `intfloat/e5-*` families — buy accuracy
at several times the cost. Start here; upgrade when a measurement tells you to, not before.

**256 tokens of context.** The number people forget, and it ruins results silently. Anything past token 256
is **truncated and gone**. Embed a ten-page document with this model and you have embedded its first
paragraph. Long text must be chunked first and embedded chunk by chunk — exactly what tutorial 3's RAG
pipeline does, and now you know why the chunk size is what it is.

**`normalize_embeddings=True`** scales every vector to length 1, which is why the printed length is 1.0. Do
this always. With unit vectors the dot product **is** the cosine similarity, so "how similar are these two
texts" becomes one multiplication and a 120,000-document search becomes one matrix product. Sections 5 and 6
are that sentence made concrete.

On the model id: the `sentence-transformers/` prefix is the namespace, the same convention as the dataset ids
in tutorial 5. The bare name still resolves today, but namespaced ids are the portable form.

## 4. What pooling actually does

`.encode()` did three things: tokenize, run the transformer, and squash one vector *per token* into one
vector *per sentence*. That last step is **pooling**, and for this model it is a plain average over the
non-padding tokens.

Worth doing once by hand — not because you will write it again, but because a library you cannot unbundle is
a library you cannot debug.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

tok  = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
bert = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

batch = tok(zoo[:2], padding=True, truncation=True, max_length=256, return_tensors="pt")
with torch.no_grad():
    out = bert(**batch)

token_vectors = out.last_hidden_state           # (2, seq_len, 384) — one vector per token
mask = batch["attention_mask"].unsqueeze(-1)    # (2, seq_len, 1)   — 1 real, 0 padding

summed = (token_vectors * mask).sum(dim=1)      # padding positions contribute nothing
counts = mask.sum(dim=1).clamp(min=1e-9)        # how many real tokens
pooled = summed / counts                        # mean pooling
manual = torch.nn.functional.normalize(pooled, p=2, dim=1)

print("token vectors:", tuple(token_vectors.shape))
print("pooled       :", tuple(manual.shape))
print("largest difference vs .encode():", float((manual - torch.tensor(Z[:2])).abs().max()))

### Code walkthrough — the mask is the whole trick

**The difference prints as ~1e-7**, which is float noise. `.encode()` is the twelve lines above, plus
batching and a progress bar.

**Multiplying by the attention mask before summing is not optional.** Padding tokens have hidden states —
not zeros, but whatever the model computed for `[PAD]` in that context. Average them in and a short sentence
in a batch of long ones has its meaning diluted by nonsense, *and the amount of dilution depends on what else
happened to be in the batch*. The same sentence would embed differently on Tuesday. This is tutorial 1's
silent-corruption failure wearing a different hat, and it is the number-one bug in hand-rolled embedding code.

**Mean pooling is a choice, not a law.** BERT-style models also expose a `[CLS]` token, which is a good
sentence vector *only if the model was trained to make it one*. Vanilla `bert-base` was not, which is why
`[CLS]` from an untuned BERT is a famously mediocre sentence embedding. The sentence-transformers checkpoints
were fine-tuned with a contrastive objective — millions of (similar, dissimilar) pairs pulled together and
pushed apart — with the pooling step **inside** the training loop. Weights and pooling were trained as one
unit; swap the pooling afterwards and you are using the model off-label.

**So: match the pooling to the checkpoint.** `.encode()` reads it from the model's config, which is the real
argument for using the library rather than the twelve lines.

## 5. Cosine similarity, and sentences the model cannot tell apart

Normalized vectors turn the whole similarity question into `Z @ Z.T`. Here are all 64 pairs at once.

In [ ]:
import numpy as np

S = Z @ Z.T          # cosine similarity, because every row has length 1

fig, ax = plt.subplots(figsize=(7.4, 6.2))
im = ax.imshow(S, cmap=BLUES, vmin=0, vmax=1)

short = ["cat: vase", "cat: curtains", "cat: purring",
         "space: rover", "space: docking",
         "cook: egg whites", "cook: dough", "cook: sauce"]
ax.set_xticks(range(8), short, rotation=45, ha="right")
ax.set_yticks(range(8), short)

for i in range(8):
    for j in range(8):
        ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="#ffffff" if S[i, j] > 0.55 else INK_SOFT)

ax.set_title("Cosine similarity — the bright blocks are the topics",
             loc="left", pad=12, color=INK)
fig.colorbar(im, ax=ax, shrink=0.8, label="cosine similarity")
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()

### Reading the heatmap

Three blocks on the diagonal — cats, space, cooking — and near-white everywhere else. Nobody told the model
what the topics were. Nobody told it what a topic is.

Three details worth pausing on:

**The diagonal is exactly 1.00.** Every sentence is perfectly similar to itself. If yours is not, your
vectors are not normalized.

**The same-topic blocks are not bright.** Two sentences about cats score around 0.3–0.45, nowhere near 1.
Beginners expect "similar" to mean 0.9 and conclude the model is broken. It is not — different sentences
about a shared topic are genuinely only *somewhat* alike, and the number that matters is the **contrast**
between same-topic and different-topic, not the absolute value.

**Off-block values sit around 0.05–0.15, not −1.** Cosine similarity spans [−1, 1] in principle, but real
sentence embeddings rarely go negative: the model's outputs occupy a narrow cone of the space, so unrelated
text scores near zero and *opposite* text does not score −1. The consequence is that **there is no universal
threshold**. 0.45 is "clearly related" for this model and might be "barely related" for another. Calibrate
against pairs you have judged yourself, every time you change models.

In [ ]:
traps = [
    ("The cat sat on the mat.",        "The mat sat on the cat."),
    ("I love this restaurant.",        "I don't love this restaurant."),
    ("The film was a total disaster.", "The film was a complete triumph."),
    ("Flight AA221 lands at 09:40.",   "Flight AA221 lands at 21:40."),
]

flat = [s for pair in traps for s in pair]
T = encoder.encode(flat, normalize_embeddings=True)

print("similarity   sentence pair")
for k, (a, b) in enumerate(traps):
    sim = float(T[2 * k] @ T[2 * k + 1])
    print(f"     {sim:.3f}   {a}")
    print(f"             {b}")

### Code walkthrough — the blind spots, and why they are not bugs

Read the scores against the right baseline. Two sentences from *different topics* in the heatmap scored about
**0.07**. Every pair here scores far above that — 0.97 for the reversed sentence, 0.94 for the wrong time,
0.78 for the negation, and even the antonym pair, the lowest of the four, lands near 0.58. Now read the pairs
again. Each means something **different**, and two of them mean the **opposite**.

- **Word order.** "The cat sat on the mat" and "The mat sat on the cat" contain identical words. Mean pooling
  averages token vectors, and averaging is order-blind. Attention does see position, so the token vectors
  differ a little — but the average largely washes that out.
- **Negation.** "I don't love this" sits close to "I love this": `don't` is one short token among many, and
  topic, register and vocabulary are otherwise identical. Embeddings encode *aboutness* far more strongly
  than *polarity*.
- **Antonyms.** *Disaster* and *triumph* occur in near-identical contexts throughout the training corpus, so
  distributional semantics places them near each other for precisely the reason it works at all. This is the
  pair the model separates best of the four — partial credit — and it is still a long way from "unrelated".
- **Numbers.** 09:40 and 21:40 differ by a token or two and not at all in meaning-space.

**None of this is a defect.** A sentence embedding is a lossy summary optimized for *topical* similarity, and
it is superb at that. The mistake is asking it for something else. Two consequences you will actually hit:

1. **Do not build a sentiment classifier out of raw embedding similarity.** The trained classifier in
   section 15 is fine — it learns which directions in the space carry polarity — but "similarity to the word
   *positive*" is not. That is the difference between a learned projection and a vibe.
2. **Do not use embedding search for exact-match questions.** "Which flight lands at 21:40" is a job for a
   filter, an index, or SQL. Hybrid retrieval — BM25 for the literal, embeddings for the semantic — exists
   because neither half is sufficient alone.

*Try it yourself:* add a pair the model **should** separate and you suspect it will not. Dates, quantities,
proper names and code identifiers are fertile ground.

## 6. Semantic search in two lines

The shortest useful thing an embedding buys you: search that matches meaning rather than characters. Watch
that the queries below share **no words** with the sentences they retrieve — and read the first result
carefully, because it is not the one you expect.

In [ ]:
def search(query, corpus, corpus_emb, k=3):
    q = encoder.encode([query], normalize_embeddings=True)[0]
    scores = corpus_emb @ q                      # one dot product per document
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), corpus[i]) for i in top]

for q in ["something to eat", "a naughty pet", "outer space"]:
    print(f"query: {q}")
    for score, text in search(q, zoo, Z):
        print(f"   {score:.3f}  {text}")
    print()

### Code walkthrough — this is what a vector database does

`corpus_emb @ q` is the entire retrieval engine. Chroma, FAISS and pgvector add persistence, metadata
filters, and an *approximate* nearest-neighbour index that keeps search sublinear at a billion vectors — but
the scoring function is this line. Tutorial 3's retriever, unwrapped.

**Brute force lasts longer than people expect.** 100,000 × 384 floats is 150 MB, and the matrix product takes
milliseconds. Reach for an index when you outgrow memory, not when you outgrow elegance.

**`np.argsort(-scores)`** sorts descending. On a large corpus use `np.argpartition`, which finds the top *k*
without bothering to order the rest.

**The queries share no vocabulary with their matches.** Not one of them appears verbatim in the sentence it
retrieves. "A naughty pet" finds the cats without the words *naughty* or *pet* occurring anywhere. That is
the value proposition of dense retrieval over keyword search.

**And now the fine print, which the first query hands us for free.** "Something to eat" ranks *"Purring is a
cat's way of asking for food"* **above** the three cooking sentences. It is not wrong, exactly: that sentence
is the one most *about* food, and the cooking sentences are about technique — folding, resting, simmering.
Dense retrieval ranks by topical similarity, and the topic here is food, not cooking. Change the query to
"how do I bake something" and the ranking flips.

Two lessons in one result. **Queries are prompts** — phrase them the way the document you want would phrase
itself, exactly as with the label descriptions in section 14. And **scores are relative**: 0.43 is the top
hit here, which would be an unremarkable score elsewhere. Any absolute cutoff you hard-code ("keep hits above
0.5") will silently return nothing the first time you change models or domains.

## 7. A real corpus: 120,000 news headlines, labels hidden

The zoo was a toy. Now: **AG News** — short news items in four categories. We take a slice, and then commit
to a small piece of theatre that is also good practice.

**We hide the labels.** Sections 8 and 9 pretend the `label` column does not exist, exactly as it would not for
the support tickets, survey answers or chat logs you were actually handed. The labels come back out in section 10, purely
to grade the clustering.

In [ ]:
import collections
from datasets import load_dataset

RAW = load_dataset("fancyzhx/ag_news")
LABEL_NAMES = RAW["train"].features["label"].names
print(RAW)
print("\nlabel names:", LABEL_NAMES)

pool = RAW["train"].shuffle(seed=0).select(range(3000))   # unlabelled corpus + labelled pool later
test = RAW["test"].shuffle(seed=0).select(range(2000))    # untouched until section 14

texts      = pool["text"]
true_label = np.array(pool["label"])        # locked in a drawer until section 10
test_texts = test["text"]
test_label = np.array(test["label"])

print("\npool balance:", dict(collections.Counter(true_label)))
print("test balance:", dict(collections.Counter(test_label)))
print("\nexample:", texts[0][:300])

### Code walkthrough — the dataset id, and the shuffle (again)

**`fancyzhx/ag_news`**, not `ag_news`. Current `huggingface_hub` versions want `namespace/name`, the same
change that bit tutorial 5's `rotten_tomatoes`.

**`.shuffle(seed=0)` before `.select(...)`**, for the same reason as tutorial 5: many HF datasets ship
grouped by label, and an unshuffled slice can be one class repeated. Print the balance and look at it. Here
the four classes come out close to even, which matters — k-means has a documented weakness for unequal
cluster sizes, and it is useful to see it succeed on the easy case before section 19 lists how it fails.

**Two splits, two jobs.** `pool` is the corpus we cluster and, later, the pool we draw training examples
from. `test` is touched for the first time in section 14 and never trained on. Keeping that boundary in a
notebook takes deliberate effort — everything is in scope, everything is one cell away — and leakage through
a stray variable is the classic way a notebook reports 0.99 and a deployment reports 0.71.

## 8. Embed the corpus

One call, 5,000 texts. Time it: the number is the entire operating cost of everything that follows.

In [ ]:
import time

t0 = time.perf_counter()
E      = encoder.encode(texts,      normalize_embeddings=True, batch_size=64, show_progress_bar=True)
E_test = encoder.encode(test_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
elapsed = time.perf_counter() - t0

n = len(texts) + len(test_texts)
print(f"\nembedded {n:,} texts in {elapsed:.1f}s  ({n / elapsed:,.0f} texts/second)")
print("corpus matrix:", E.shape, "=", f"{E.nbytes / 1e6:.1f} MB")

### Code walkthrough — the cost model of this notebook

**Around a hundred texts per second on a laptop CPU; thousands on a GPU.** That makes the whole of AG News
about twenty minutes of CPU time, once — and every experiment afterwards is linear algebra on a 180 MB array.
That asymmetry is the reason to reach for embeddings first: the expensive step happens a single time, and
clustering, search and classification all reuse the result.

**`batch_size=64`** trades memory for throughput. Raise it on a GPU until memory complains; it changes speed
only, never results.

**Embeddings are a cache, so treat them like one.** In a real project, write `E` to disk
(`np.save("emb.npy", E)`) keyed by *both* the corpus version and the model name. Re-embedding on every run is
the most common avoidable cost in this kind of pipeline — and mixing vectors from two different models in one
index produces silent garbage, since their spaces are unrelated.

**One warning about `.encode()` on a list of raw strings:** some newer models want an instruction prefix
(`"query: "` / `"passage: "` for `e5`, a task prompt for `bge`). Skip it and you lose several points of
accuracy with no error message. Check your model's card; MiniLM needs nothing.

## 9. k-means: four groups, no labels

Now the unsupervised part. We ask for four clusters — a choice we interrogate in section 11 — and the
algorithm does the rest: pick 4 centres, assign every point to its nearest, move each centre to the mean of
what it caught, repeat until nothing moves.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=0)
cluster = km.fit_predict(E)

print("cluster sizes:", dict(sorted(collections.Counter(cluster).items())))
print("centroid matrix:", km.cluster_centers_.shape)
print()

for c in range(4):
    members = np.where(cluster == c)[0]
    centre = km.cluster_centers_[c] / np.linalg.norm(km.cluster_centers_[c])
    closest = members[np.argsort(-(E[members] @ centre))[:3]]
    print(f"--- cluster {c}  ({len(members)} documents)")
    for i in closest:
        print("   ", texts[i][:110].replace("\n", " "))

### Code walkthrough — three details that decide whether this works

**`n_init=10`.** k-means is not deterministic: it depends on where the centres start, and it converges to a
*local* optimum. Running it ten times from different starts and keeping the best (lowest inertia) is not a
luxury. With `n_init=1` you are reporting one roll of the dice.

**Euclidean distance on normalized vectors is cosine distance in disguise.** scikit-learn's k-means minimizes
squared Euclidean distance, but for unit vectors ‖a−b‖² = 2 − 2·(a·b), which decreases exactly as cosine
similarity increases. So normalizing in section 8 quietly turned this into *spherical* k-means, the variant
you actually want for text. Without normalization, long documents get long vectors and k-means starts
clustering by **document length**. This is the single highest-value line in the notebook.

**Printing the three documents nearest each centroid** is how you inspect a clustering. The centroid is an
average of vectors and therefore not any real document; the documents closest to it are the readable stand-in.
Do this before you compute a single metric — if the nearest-centroid documents look like nothing in
particular, no score will save the result.

Four clusters, and each one's exemplars are visibly about one thing. Nobody supplied a taxonomy.

## 10. Grading a clustering, honestly

Now we open the drawer and compare the clusters with the true categories.

The temptation is to call this *accuracy*. That is wrong, and the reason matters: **cluster numbers are
arbitrary**. Cluster 2 might be Sports; re-run with a different seed and Sports becomes cluster 0. A correct
clustering can score 0% accuracy purely from permutation. Metrics for this job must be invariant to
relabelling.

In [ ]:
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

grid = pd.crosstab(pd.Series(cluster, name="cluster"),
                   pd.Series([LABEL_NAMES[i] for i in true_label], name="true category"))
print(grid, "\n")

ari = adjusted_rand_score(true_label, cluster)
nmi = normalized_mutual_info_score(true_label, cluster)

# Best one-to-one cluster -> category matching, then how often it is right.
row, col = linear_sum_assignment(-grid.values)
mapping = {int(r): grid.columns[c] for r, c in zip(row, col)}
matched = grid.values[row, col].sum() / grid.values.sum()

print(f"Adjusted Rand Index      {ari:.3f}   (0 = chance, 1 = identical partition)")
print(f"Normalized Mutual Info   {nmi:.3f}   (0 = independent, 1 = one determines the other)")
print(f"Best-matching agreement  {matched:.3f}")
print("\nbest matching:", mapping)

### Code walkthrough — what each number means, and when to believe it

**The crosstab is the real result**; the scalars summarise it. Read it column by column: a category spread
across two clusters means the model found a *finer* distinction than the taxonomy (Sci/Tech splitting into
science and gadgets, say). A cluster absorbing part of a second category means it found a coarser one. Look
for the off-diagonal cells: a sizeable share of Sci/Tech lands with Business (corporate technology news is
both) and another share lands with World. Those overlaps are in the language, not in the algorithm — and the
classifier in section 15 will trip on exactly the same boundary.

**Adjusted Rand Index** asks: over all pairs of documents, how often do the clustering and the truth agree on
"same group / different group"? *Adjusted* means corrected for chance, so random labelling scores ≈ 0 rather
than the ≈ 0.25 raw agreement would give. It can go negative (worse than random). ARI is the strict one, and
it punishes splitting a true class in half.

**Normalized Mutual Information** asks how much knowing the cluster tells you about the category, in bits,
normalized to [0, 1]. It is more forgiving of splits — a class cleanly divided into two clusters still
carries full information about the class. Report both: NMI far above ARI is the signature of exactly that
split.

**Best-matching agreement** is the closest honest thing to accuracy. `linear_sum_assignment` solves the
assignment problem — the optimal one-to-one matching of clusters to categories — and then we count. It is
readable ("83% of documents landed in the right group"), but it only exists because we *have* labels, and if
we had labels we would have trained a classifier. Treat it as a diagnostic, never as a deliverable.

**The honest summary of these numbers:** this is unsupervised learning recovering most of a human taxonomy
from nothing but the text, which is genuinely impressive — and it is comfortably worse than the classifier in
section 15, which sees a hundred labels per class. Clustering is for *discovering* categories, not for
predicting known ones.

## 11. Choosing k when nobody tells you k

We asked for four because we had peeked. In real work the number of groups is the thing you least know.
There is no correct answer, but there are two useful instruments.

In [ ]:
from sklearn.metrics import silhouette_score

ks, sils, inertias = range(2, 11), [], []
for k in ks:
    kmk = KMeans(n_clusters=k, n_init=10, random_state=0).fit(E)
    sils.append(silhouette_score(E, kmk.labels_, metric="cosine", sample_size=2000, random_state=0))
    inertias.append(kmk.inertia_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.plot(list(ks), sils, color=SERIES[0], marker="o", markersize=7)
ax1.set_title("Silhouette — higher is better", loc="left", color=INK, pad=10)
ax1.set_xlabel("number of clusters (k)")
ax1.set_ylabel("mean silhouette")
best = int(np.argmax(sils))
ax1.annotate(f"peak at k={list(ks)[best]}", (list(ks)[best], sils[best]),
             textcoords="offset points", xytext=(8, 8), color=INK_SOFT, fontsize=9)

ax2.plot(list(ks), inertias, color=SERIES[1], marker="s", markersize=7)
ax2.set_title("Inertia — look for the elbow", loc="left", color=INK, pad=10)
ax2.set_xlabel("number of clusters (k)")
ax2.set_ylabel("within-cluster sum of squares")

for ax in (ax1, ax2):
    ax.grid(axis="y")
    ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

print("silhouette by k:", {k: round(s, 3) for k, s in zip(ks, sils)})

### Code walkthrough — two instruments, and why they are on separate axes

**Silhouette** scores each point by (distance to its own cluster) versus (distance to the nearest other
cluster), averaged over the corpus. Near 1 means tight, well-separated clusters; near 0 means points sit on
boundaries; negative means they are in the wrong cluster. Pass `metric="cosine"` so the score is measured the
same way the clustering was, and `sample_size` because the exact computation is O(n²).

**Inertia** is what k-means minimizes: total squared distance to centroids. It *always* falls as k rises —
at k = n it is zero — so its minimum is meaningless. You look for the **elbow**, the point where buying
another cluster stops paying. Elbows are notoriously in the eye of the beholder; `kneed` will locate one
mechanically if you want it reproducible.

**Two panels, not two y-axes.** Silhouette runs 0–1 and inertia runs in the hundreds. Overlaying them on a
twin axis would let the shared x-axis imply crossings and coincidences that are artefacts of two arbitrary
scales — the most common way a chart lies. Two panels side by side compare just as well and cannot mislead.

**Now the disappointing part — and it is the point of the section.** Look at the actual values: every one of
them is somewhere around 0.05–0.08. On a scale where 1 is "perfectly separated" and 0 is "sitting on the
boundary", **this clustering scores almost zero** — the same clustering that section 10 will show recovers
83% of a human taxonomy. Both facts are true. High-dimensional text embeddings form clouds that touch, not
islands, so silhouette values in the low tenths are normal and near-zero is not a failure. If you go looking
for the 0.7 you saw in a textbook example on two-dimensional blobs, you will reject a perfectly good result.

And the peak is very unlikely to land on 4. It tends to sit at 2 or 3, because at the coarsest level this
corpus really does split into something like "money and technology" versus "people and events", and
geometrically that split is cleaner than the four-way one. **Silhouette measures geometry, not usefulness.**
Real guidance:

- use the curve to narrow the range, then *read the clusters* at two or three candidate k values;
- let the downstream use decide — if a human triages the clusters, more than ~15 is unmanageable regardless
  of the score;
- if you want the algorithm to choose, use one that can: **HDBSCAN** finds density-based clusters, picks
  their number itself, and — unusually and usefully — labels genuine outliers `-1` instead of forcing every
  document into a group.

## 12. What is this cluster *about*? Two ways to find out

A cluster is a set of row indices. To act on it, somebody has to name it. Doing that by hand across 40
clusters is a day's work, so here are both automatic routes: statistics first, then the LLM.

The statistical route is **c-TF-IDF** — the idea behind BERTopic. Glue every document in a cluster into one
giant document, then ask which words are frequent *here* and rare *elsewhere*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

merged = [" ".join(np.array(texts, dtype=object)[cluster == c]) for c in range(4)]
vec = TfidfVectorizer(stop_words="english", max_features=30000, sublinear_tf=True)
M = vec.fit_transform(merged)
vocab = np.array(vec.get_feature_names_out())

keywords = {}
for c in range(4):
    scores = M[c].toarray().ravel()
    keywords[c] = list(vocab[np.argsort(-scores)[:10]])
    print(f"cluster {c:>2}  ({(cluster == c).sum():>4} docs)  {', '.join(keywords[c])}")
    print(f"           true majority: {mapping[c]}")

### Code walkthrough — why merging first is the whole idea

**One row per cluster, not per document.** Standard TF-IDF asks "which words distinguish this *document*?"
Concatenating first changes the question to "which words distinguish this *cluster*?" — which is what a name
needs. That substitution is all c-TF-IDF is.

**`stop_words="english"`** removes *the*, *of*, *said*. Without it, every cluster's top terms are identical
function words and the output is worthless. For other languages, supply your own list — this one is English-only.

**`sublinear_tf=True`** replaces raw counts with 1 + log(count), so a word appearing 900 times does not
outweigh one appearing 90 times by tenfold. In concatenated documents, where counts are huge and skewed, this
matters more than usual.

The keyword lists are readable enough to name the clusters by hand. They are also unreadable enough that you
would rather not do it forty times — which is the cue for the next cell.

### The fun version: let an LLM read the evidence and name the cluster

Keywords plus a few exemplar headlines are exactly the sort of thing a language model turns into a crisp
label. This is the pattern worth remembering: **the statistics find the evidence, the LLM writes the name.**

The cell below reuses the Groq setup from tutorials 3 and 4 — an `.env` file next to this notebook with
`GROQ_API_KEY=...`. It is optional; the notebook continues without it.

In [ ]:
# Optional — needs GROQ_API_KEY in a .env file beside this notebook.
# pip install langchain-groq python-dotenv
try:
    from dotenv import load_dotenv
    from langchain_groq import ChatGroq

    load_dotenv()
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

    for c in range(4):
        members = np.where(cluster == c)[0]
        centre = km.cluster_centers_[c] / np.linalg.norm(km.cluster_centers_[c])
        examples = [texts[i][:160] for i in members[np.argsort(-(E[members] @ centre))[:5]]]

        prompt = (
            "These documents form one cluster of a news corpus.\n\n"
            f"Distinctive keywords: {', '.join(keywords[c])}\n\n"
            "Representative documents:\n"
            + "\n".join(f"- {e}" for e in examples)
            + "\n\nGive a 2-4 word topic label for this cluster. Reply with the label only."
        )
        print(f"cluster {c}: {llm.invoke(prompt).content.strip():<28} (true majority: {mapping[c]})")

except Exception as exc:
    print("Skipping the LLM naming step:", type(exc).__name__, exc)
    print("Fall back on the keyword lists above — they are usually enough.")

### Code walkthrough — and one caution

**Why the evidence is chosen this way.** Keywords give the LLM the *distinctive* vocabulary; the
nearest-centroid documents give it the *typical* content. Either alone produces worse names — keywords alone
lack context, documents alone tempt the model to describe one story rather than the group.

**`temperature=0`** because this is labelling, not writing. You want the same cluster to get the same name
on every run.

**"Reply with the label only"** because the default behaviour of a chat model is to be helpful at length, and
"Certainly! Here is a label for your cluster:" is not a column value. Tutorial 3's structured-output tools
are the robust version of this instruction.

**Five documents out of eight hundred.** The model names the cluster from the sample you chose, and it will
name it confidently either way. If the exemplars are unrepresentative, the label is wrong and nothing in the
output says so. Sample from a few different distances from the centroid, and spot-check names against the
crosstab — which, for once, we have.

**This scales the way the manual version does not.** Forty clusters is forty cheap calls. That is the real
argument for the pattern: not that the LLM names better than you, but that it names at 3 a.m. across every
cluster in the pipeline.

## 13. Draw the map — carefully

384 dimensions do not fit on a page, so we project to 2. **t-SNE** arranges points so that near-neighbours in
384-D stay near-neighbours in 2-D, and lets everything else fall where it may.

Two panels, same coordinates: coloured by what k-means found, and by what the labels say. The left panel is
what you would have without labels; the right is the answer key.

In [ ]:
from sklearn.manifold import TSNE

idx = np.random.default_rng(0).choice(len(E), size=1200, replace=False)
XY = TSNE(n_components=2, metric="cosine", init="pca",
          perplexity=30, learning_rate="auto", random_state=0).fit_transform(E[idx])

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharex=True, sharey=True)

panels = [("Found by k-means (no labels)", cluster[idx], [f"cluster {c}" for c in range(4)]),
          ("The actual categories",        true_label[idx], LABEL_NAMES)]

for ax, (title, groups, names) in zip(axes, panels):
    for g in range(4):
        sel = groups == g
        ax.scatter(XY[sel, 0], XY[sel, 1], s=14, alpha=0.75,
                   c=SERIES[g], marker=MARKERS[g], linewidths=0, label=names[g])
    ax.set_title(title, loc="left", color=INK, pad=10)
    ax.set_xticks([]), ax.set_yticks([])
    ax.legend(loc="upper right", frameon=False, fontsize=9, markerscale=1.4)

fig.suptitle("Same 1,200 headlines, same projection — two colourings",
             x=0.01, ha="left", color=INK_SOFT, fontsize=10)
plt.tight_layout()
plt.show()

### Code walkthrough — and a standing warning about this plot

**`metric="cosine"`** keeps the projection consistent with how we clustered. **`perplexity=30`** is roughly
"how many neighbours each point should care about" — 5 fragments the picture, 50 smooths it into blobs; try
both, because the shapes you get are as much a property of this number as of your data. **`init="pca"`**
starts from a linear projection, which makes runs far more reproducible than the random start.

**Compare the panels, not the shapes.** Where the two colourings agree, k-means recovered a real category.
Where the left panel splits an island the right panel keeps whole, it found a sub-topic. Where colours
interleave — Business against Sci/Tech, and both against World — the categories genuinely overlap in the
text, and no clustering algorithm can separate what the language does not. Sports, by contrast, tends to sit
off on its own: the one category here whose vocabulary is unlike everything else.

**Now the warning, because this is the most over-interpreted plot in machine learning.**

- **Distances between clusters are not meaningful.** t-SNE preserves neighbourhoods, not global geometry. Two
  blobs far apart on screen are not "more different" than two blobs close together.
- **Blob sizes are not meaningful.** t-SNE expands sparse regions and compresses dense ones by construction.
- **It will show you clusters in pure noise.** Run it on random vectors and you will get islands. The picture
  is never evidence *that* clusters exist.
- **Never cluster the 2-D coordinates.** Cluster in the full space — as we did — and use the map to look at
  the result. Clustering the projection means clustering an artefact.

**UMAP** (`pip install umap-learn`) is the common alternative: faster on large corpora, better at preserving
global structure, and unlike t-SNE it can `transform()` new points into an existing projection. Every warning
above still applies.

## 14. Classification with zero training examples

Switch to supervised territory — except we have no training data. We have the *label names*, and a model that
puts similar meanings in similar places. So: embed the labels, embed the documents, and give each document
the nearest label.

This is the cheapest classifier that exists. It is also the one people get wrong first, so we run it twice.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

SCORES = {}

bare = LABEL_NAMES     # ["World", "Sports", "Business", "Sci/Tech"]

described = [
    "world news about international affairs, politics, war and events in other countries",
    "sports news about games, matches, tournaments, teams, athletes and results",
    "business news about companies, markets, stocks, earnings, trade and the economy",
    "science and technology news about research, computers, software, the internet and space",
]

for name, prompts in [("zero-shot: label names", bare),
                      ("zero-shot: label descriptions", described)]:
    L = encoder.encode(prompts, normalize_embeddings=True)
    pred = np.argmax(E_test @ L.T, axis=1)          # nearest label, per document
    SCORES[name] = accuracy_score(test_label, pred)
    print(f"{name:<32} accuracy {SCORES[name]:.3f}")

print()
print(classification_report(test_label, pred, target_names=LABEL_NAMES, digits=3))

### Code walkthrough — the gap between those two numbers is the lesson

**One matrix product classifies the whole test set.** `E_test @ L.T` is 2000×384 times 384×4. There is no
training step at all — add a fifth category by writing a fifth sentence.

**Descriptions beat bare names, usually by a lot.** "Business" as a standalone word embeds to something
vague and slightly corporate; the descriptive sentence lands in the region where *articles about markets and
earnings* actually live. You are not writing a name, you are writing **a sentence that sits where the class
sits**. Concretely: write the label the way a document in that class would describe itself, use the
vocabulary of the domain, and keep all your descriptions parallel in length and register, because a much
longer description is not penalised but a differently-*shaped* one is.

That is prompt engineering, in a system with no prompt. Which also means it has the same failure mode: you
are tuning those sentences against something, and if you tune them against the test set you have leaked. Keep
a handful of validation documents for this.

**Read the per-class report, not just the accuracy.** The report above is for the *descriptions*, and its
four classes come out reasonably even — which is what a good set of descriptions looks like. Print the same
report for the bare names and the picture is lumpier: a mean of 0.63 is not four classes at 0.63, it is
usually two decent ones and one being eaten by its neighbour. Recall far below precision for a class means
its description is losing documents to another description, and rewriting that one sentence is the whole fix.

**What this is genuinely for:** day zero, when the labels exist and the data does not. It is an honest
baseline, it takes five minutes, and it gives you something to be better than. If it already meets your bar,
stop — you have shipped a classifier with no training data.

## 15. A hundred examples change everything

Now suppose someone labels a hundred documents per class — an afternoon of work. We keep the same embeddings
and put a **logistic regression** on top: a linear model learning, per class, which directions in the
384-dimensional space point towards it.

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
few = np.concatenate([rng.choice(np.where(true_label == c)[0], 100, replace=False) for c in range(4)])

t0 = time.perf_counter()
clf = LogisticRegression(max_iter=2000, C=1.0).fit(E[few], true_label[few])
train_time = time.perf_counter() - t0

pred = clf.predict(E_test)
SCORES["trained: 100 examples/class"] = accuracy_score(test_label, pred)

print(f"trained on {len(few)} documents in {train_time:.2f}s")
print(f"accuracy {SCORES['trained: 100 examples/class']:.3f}\n")
print(classification_report(test_label, pred, target_names=LABEL_NAMES, digits=3))

print("rows = true, columns = predicted\n")
print(pd.crosstab(pd.Series([LABEL_NAMES[i] for i in test_label], name="true"),
                  pd.Series([LABEL_NAMES[i] for i in pred],       name="predicted")))

### Code walkthrough — why a linear model is enough

**Fractions of a second on a CPU.** Compare with tutorial 5: minutes on a GPU, and that was a small model on
a small dataset. The embedding did the hard work already; all that is left is drawing hyperplanes.

**A linear model, on purpose.** The transformer spent 6 layers of non-linearity arranging this space so that
meaning is *linearly* separable. Logistic regression exploits that arrangement. Gradient boosting or an MLP on
the same features will usually gain you a point at most, and cost you calibration and interpretability. Start
linear; it is a real result, not a placeholder.

**`max_iter=2000`** because the default 100 rarely converges on 384 dense features — a `ConvergenceWarning`
means the number you just printed is from a model that stopped early. **`C`** is inverse regularization
strength: lower it when you have very few examples per class, and tune it with cross-validation
(`LogisticRegressionCV`) rather than by feel.

**The confusion matrix is where the remaining errors live**, and by now they should be familiar. Sports is
nearly perfect — its vocabulary shares almost nothing with the other three. The errors concentrate on
Business against Sci/Tech, and both against World: the same boundaries the clustering blurred in section 10
and the zero-shot classifier got wrong in section 14. Three different methods failing on the same pairs is
strong evidence that the *boundary* is the problem and not the method — and the fix is a clearer taxonomy, or
more labels specifically *there*, rather than a bigger model.

**`predict_proba` gives you an abstain button.** For anything with consequences, route low-confidence
predictions to a human instead of guessing. That single line of triage is usually worth more than several
points of accuracy.

## 16. How many labels do you actually need?

The question every project asks and almost none measures. It takes about ten seconds to answer: train on
1, 5, 10, ... examples per class and plot it.

In [ ]:
sizes = [1, 2, 5, 10, 25, 50, 100, 250, 500]
curve = []
for n_per in sizes:
    accs = []
    for seed in range(5):
        r = np.random.default_rng(seed)
        take = np.concatenate([r.choice(np.where(true_label == c)[0], n_per, replace=False)
                               for c in range(4)])
        m = LogisticRegression(max_iter=2000).fit(E[take], true_label[take])
        accs.append(accuracy_score(test_label, m.predict(E_test)))
    curve.append((np.mean(accs), np.std(accs)))

mean = np.array([c[0] for c in curve])
sd   = np.array([c[1] for c in curve])
zs   = SCORES["zero-shot: label descriptions"]

fig, ax = plt.subplots(figsize=(8.2, 4.6))
ax.fill_between(sizes, mean - sd, mean + sd, color=SERIES[0], alpha=0.15, linewidth=0)
ax.plot(sizes, mean, color=SERIES[0], marker="o", markersize=7, label="embeddings + logistic regression")
ax.axhline(zs, color=SERIES[1], linestyle="--", linewidth=2, label="zero-shot (no training data)")

ax.set_xscale("log")
ax.set_xticks(sizes, [str(s) for s in sizes])
ax.set_xlabel("labelled examples per class")
ax.set_ylabel("test accuracy")
ax.set_title("The first hundred labels do most of the work", loc="left", color=INK, pad=10)
ax.annotate(f"{mean[-1]:.3f}", (sizes[-1], mean[-1]), textcoords="offset points",
            xytext=(-6, 10), color=SERIES[0], fontsize=9, ha="right")
ax.annotate(f"{zs:.3f}", (sizes[0], zs), textcoords="offset points",
            xytext=(0, 8), color=SERIES[1], fontsize=9)
ax.grid(axis="y"), ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

for s, (m, d) in zip(sizes, curve):
    print(f"{s:>4} per class   {m:.3f} ± {d:.3f}")

### Code walkthrough — reading the curve, and the shaded band

**Five draws per point, not one.** With 1 example per class, *which* example you drew matters enormously. The
shaded band is ±1 standard deviation across draws, and it is wide on the left and narrow on the right. A
single-run learning curve is mostly noise dressed as a trend, and the band is what stops you over-reading a
bump.

**The shape is the finding, and it is logarithmic.** The first handful of examples per class is worth more
than the last few hundred: expect something like a 40-point jump between 1 and 25 per class, and a gain of
one or two points between 250 and 500. The practical rule: **label a hundred per class, measure, then decide
whether labelling more is the best use of the next afternoon.** Past that point the money is usually better
spent cleaning the taxonomy or fixing the hard boundary from section 15.

**The crossover against zero-shot is the number to remember.** Somewhere around ten examples per class, the
trained model catches the label-description approach; by twenty-five it is clearly ahead. Ten labelled
examples per class is *an hour of work*, which is the real headline of this chart. Measure your own crossover
rather than borrowing this one — on a harder task it moves right, and on a task involving negation or
polarity it moves sharply left, because the zero-shot baseline it has to beat is much weaker there.

**Log x-axis, because the interesting action is at small n.** On a linear axis the left half — the half you
actually care about — is squashed against the y-axis.

**Where fine-tuning enters.** Extend this curve to thousands per class and it keeps creeping up, and a
fine-tuned DistilBERT (tutorial 5) eventually passes it, because it can adapt the *features* and not just the
boundary. On this task that crossover typically lands somewhere in the low thousands. Below it, embeddings
plus logistic regression is not a lesser option — it is the better one.

## 17. The other two contenders: NLI models and prompting an LLM

Two more ways to classify without training data. Both are much slower than a dot product, and each earns its
keep somewhere specific.

**An NLI model** was trained to judge whether a premise entails a hypothesis. Turn each label into a
hypothesis — *"This text is about sports."* — and the entailment score becomes a class score. No task-specific
training, and it understands the sentence rather than averaging it, so negation and word order survive.

The cell below is **optional and slow**: `bart-large-mnli` is a 1.6 GB download, and on a CPU it runs at a
few documents per second. It scores 400 documents, not 2,000 — and note it must run the model once per
(document, label) pair, which is the cost story in one sentence.

In [ ]:
# Optional. 1.6 GB download; a few minutes on CPU. Skip freely.
RUN_NLI = True

if RUN_NLI:
    from transformers import pipeline

    zshot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    hypotheses = ["world news", "sports", "business and finance", "science and technology"]

    sub = 400
    t0 = time.perf_counter()
    out = zshot(list(test_texts[:sub]), candidate_labels=hypotheses,
                hypothesis_template="This text is about {}.", batch_size=8)
    nli_time = time.perf_counter() - t0

    pred = np.array([hypotheses.index(o["labels"][0]) for o in out])
    SCORES["zero-shot: NLI model"] = accuracy_score(test_label[:sub], pred)

    print(f"accuracy on {sub} documents: {SCORES['zero-shot: NLI model']:.3f}")
    print(f"{nli_time:.1f}s  =  {sub / nli_time:.1f} documents/second "
          f"({sub * len(hypotheses)} forward passes)")
else:
    print("skipped")

### Code walkthrough — and the cost comparison that should drive the choice

**`hypothesis_template`** is the prompt, and it matters as much as the label descriptions in section 14.
"This text is about {}." suits topics; "This complaint is about {}." or "The sentiment of this review is
{}." suit other tasks. Change it and the accuracy moves.

**One forward pass per (document, label).** Four labels means four passes per document. Twenty labels means
twenty. The embedding approach runs the model **once** per document and then does a dot product per label —
which is why it scales to hundreds of classes and this does not.

**A smaller option:** `MoritzLaurer/deberta-v3-base-zeroshot-v2.0` is roughly a quarter the size, trained
specifically for zero-shot classification, and usually as good or better. Try it as a drop-in replacement.

**And the third contender — prompting a generative LLM.** You already have the tooling from tutorials 3 and 4:
send the document and the label list, ask for one word back, parse it. It is the most flexible option by a
wide margin — it handles labels that need reasoning, it can be given rules and exceptions in prose, and it
will explain its answer. It is also, per document, thousands of times more expensive than a dot product, and
its output is free text that you must constrain and validate. Use structured outputs, and put a cache in
front of it.

Here is the whole field on one page:

| Approach | Labels needed | Per-document cost | Scales to many classes | Explains itself |
|---|---|---|---|---|
| Embedding + nearest label | 0 | one encode + a dot product | yes, trivially | no |
| Embedding + logistic regression | ~10–1000 | one encode + a dot product | yes | weakly (coefficients) |
| NLI zero-shot | 0 | one pass **per label** | poorly | no |
| Prompt an LLM | 0 | an API call, tokens in and out | yes | yes, in prose |
| Fine-tuned classifier | 1000+ | one pass | yes | no |

**A pattern worth stealing:** use the expensive option to *create* labels, then the cheap one to serve them.
Have an LLM label 2,000 documents overnight, train a logistic regression on the embeddings, and ship the
model that costs a millisecond. You get most of the LLM's quality at a fraction of the runtime cost — this is
distillation, and it is what most production classifiers built since 2023 actually are.

## 18. The scoreboard

Everything we measured, on one axis.

In [ ]:
order = sorted(SCORES.items(), key=lambda kv: kv[1])
names = [k for k, _ in order]
vals  = [v for _, v in order]

fig, ax = plt.subplots(figsize=(8.6, 0.62 * len(order) + 1.6))
ax.barh(names, vals, color=SERIES[0], height=0.62)
for y, v in enumerate(vals):
    ax.text(v + 0.012, y, f"{v:.3f}", va="center", color=INK_SOFT, fontsize=9)

ax.set_xlim(0, 1.0)
ax.set_xlabel("test accuracy")
ax.set_title("AG News, 4 classes — chance is 0.25", loc="left", color=INK, pad=10)
ax.grid(axis="x"), ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

print(f"clustering, for reference: ARI {ari:.3f}, NMI {nmi:.3f}, "
      f"best-matching agreement {matched:.3f} — and it used no labels at all")

### Reading the scoreboard

**One series, so no legend** — the title says what the bars are — and a value printed at the end of each bar,
because a reader who wants the number should not have to measure against the grid.

**The x-axis starts at zero.** A bar chart encodes value as length; start the axis at 0.6 and a three-point
difference looks like a landslide. This is the most common deliberate chart lie, and it is worth being unable
to draw.

**What the ranking says.** Zero-shot with descriptions is respectable for something that took five minutes.
A hundred labels per class beats every zero-shot method, usually by a wide margin. And the clustering — which
is not on the chart, because it never saw a label and is measured on a different scale entirely — recovered
most of the taxonomy from nothing.

**Do not read this as a universal ranking.** AG News is topic classification, which is exactly what sentence
embeddings are best at. On a task that turns on *polarity* or *negation* — sentiment, contract clauses,
safety triage — the zero-shot embedding row drops sharply (section 5 told you why), the NLI and LLM rows hold
up much better, and the trained row stays strong because it learns the directions that carry polarity. Run
your own version of this chart on your own task. That is the deliverable of this notebook, not the numbers.

## 19. Pitfalls

**Forgetting to normalize.** Without unit vectors, k-means clusters partly by document length and "cosine
similarity" is not cosine similarity. One argument, and the most consequential in the notebook.

**Mixing embeddings from two models.** Vectors from MiniLM and mpnet are not comparable — different spaces,
different dimensions, no meaningful similarity between them. Re-embed the whole corpus when you change models,
and store the model name next to the vectors.

**Silent truncation.** 256 tokens in, the rest ignored, no warning. Check your texts' lengths against the
model's `max_seq_length` before you trust anything downstream.

**Treating cluster numbers as classes.** Cluster 2 is not "Sports"; it is a set of row indices whose number
changes with the random seed. Name clusters explicitly (section 12) and keep the mapping somewhere.

**Judging a clustering by accuracy.** Use ARI and NMI, which are invariant to relabelling, and read the
crosstab before either.

**Believing the t-SNE picture.** Between-cluster distances, blob sizes and apparent islands are all artefacts.
Never cluster the 2-D coordinates.

**Taking the silhouette peak as the number of groups.** It measures geometry, not usefulness, and it loves
k = 2. Use it to narrow the range, then read the clusters.

**k-means on data that is not blobby.** It assumes roughly spherical, roughly equal-sized clusters and forces
every point into one, outliers included. For uneven sizes, odd shapes or genuine noise, use HDBSCAN.

**Ignoring class imbalance.** 95% accuracy on a 95/5 split can mean "always predict the majority". Macro-F1
and a confusion matrix will say so — the same warning as tutorial 5, and it applies to every row of the
scoreboard.

**Tuning label descriptions on the test set.** It feels like prompt engineering rather than training, so
nobody counts it. It is training, and it leaks. Hold out a validation set.

**Assuming English.** MiniLM is English-only; feed it Persian, Arabic or Chinese and you get confident
nonsense rather than an error. Use a multilingual encoder — `paraphrase-multilingual-MiniLM-L12-v2`,
`intfloat/multilingual-e5-base` — and re-run section 5's sanity check in your language before trusting it.

### Mini-lab

Take a corpus of your own — support tickets, paper abstracts, news in your language, your own chat export —
and run the whole loop:

1. embed it, and check the token-length distribution against the model's limit *before* anything else;
2. build a five-sentence Sentence Zoo for your domain, including one pair you believe the model will wrongly
   call similar. Were you right?
3. cluster at three values of k, read the c-TF-IDF keywords at each, and pick one by reading rather than by
   silhouette;
4. name the clusters — by hand and with an LLM — and note where the two disagree and why;
5. write label descriptions for the categories you found, and measure zero-shot accuracy on 50 documents you
   label by hand;
6. label 50 more, train a logistic regression, and plot the two points against each other;
7. write three sentences: which approach you would ship, what it costs per document, and what would have to
   change for you to switch.

Step 7 is the point. Every technique here works; choosing between them on evidence is the skill.

## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Embedding** | A fixed-length vector representing a text, positioned so that similar meanings are near each other. |
| **Pooling** | Collapsing one vector per token into one per text. Mean pooling over the attention mask, for most sentence models. |
| **Normalization** | Scaling a vector to length 1, so that the dot product equals cosine similarity. |
| **Cosine similarity** | Angle-based similarity in [−1, 1]. In practice, near 0 for unrelated text — not −1. |
| **Bi-encoder** | Encodes each text independently, so vectors can be cached and compared cheaply. What we used throughout. |
| **Cross-encoder** | Encodes a *pair* together for a far better score, at one forward pass per pair. Used to re-rank a bi-encoder's top hits. |
| **k-means** | Partitions points into k groups around centroids. On normalized vectors, equivalent to spherical k-means. |
| **Inertia** | k-means' objective: total squared distance to centroids. Always falls as k rises — look for the elbow, not the minimum. |
| **Silhouette** | Per-point cluster tightness versus separation, averaged. Geometry, not usefulness. |
| **HDBSCAN** | Density-based clustering: picks its own number of clusters and labels outliers instead of forcing them in. |
| **ARI** | Adjusted Rand Index — pairwise agreement with a reference partition, corrected for chance. Invariant to relabelling. |
| **NMI** | Normalized Mutual Information — how much the cluster tells you about the class. More forgiving of splits than ARI. |
| **c-TF-IDF** | TF-IDF over concatenated clusters: the terms that distinguish a *cluster*, not a document. The core of BERTopic. |
| **t-SNE / UMAP** | Non-linear projections to 2-D for viewing. Preserve neighbourhoods; distances and sizes are artefacts. |
| **Zero-shot classification** | Labelling with no training examples — by embedding label descriptions, by NLI entailment, or by prompting. |
| **NLI / entailment** | Judging whether a premise implies a hypothesis; repurposed as a classifier, one pass per label. |
| **Distillation (the practical kind)** | Labelling data with an expensive model, then training a cheap one on it. What most shipped classifiers are. |

---

**Where this sits in the course.** Tutorial 3 used embeddings for retrieval; this notebook used the same
vectors for structure and for labels. Tutorial 5 trains the model that beats everything here — once you have
the thousands of labels it needs. The order to work in is the order of this notebook: look, cluster, label a
hundred, measure, and fine-tune only when the measurement asks for it.